In [16]:
import pandas as pd

fuel = pd.read_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/fuel_prices_clean.csv",
    parse_dates=["ngay"]
)
fuel.head()

,ngay,mat_hang_clean,gia_moi
0,2012-08-01,"Dầu DO 0,05S",20800.0
1,2012-08-01,Dầu hỏa,20650.0
2,2012-08-01,"Mazút No2B (3,0S)",18450.0
3,2012-08-01,"Mazút No2B (3,5S)",18150.0
4,2012-08-01,Xăng RON 95,22400.0


## Lựa chọn mặt hàng nhiên liệu

Đề tài sử dụng giá Xăng RON 95 và dầu Diesel làm các biến đại diện cho biến động giá nhiên liệu có liên quan trực tiếp đến chi phí giao thông. Các mặt hàng nhiên liệu khác không được sử dụng trong bước xây dựng dữ liệu đầu vào.

In [17]:
fuel_selected = fuel[
    fuel["mat_hang_clean"].str.contains(
        r"Xăng RON 95|Dầu DO",
        regex=True,
        na=False
    )
].copy()
fuel_selected["mat_hang_clean"].value_counts()

mat_hang_clean
Dầu DO 0,05S        279
Xăng RON 95-III     205
Xăng RON 95         105
Dầu DO 0,001S-V      40
Xăng RON 95-II       17
Dầu DO 0,005S-IV     10
Xăng RON 95-IV       10
Name: count, dtype: int64

In [18]:
fuel_selected.groupby("mat_hang_clean").agg(
    tu_ngay=("ngay", "min"),
    den_ngay=("ngay", "max"),
    so_quan_sat=("gia_moi", "count")
).sort_values("tu_ngay")

,tu_ngay,den_ngay,so_quan_sat
mat_hang_clean,,,
"Dầu DO 0,05S",2012-08-01,2024-12-26,279
Xăng RON 95,2012-08-01,2018-12-21,105
Xăng RON 95-II,2016-04-05,2016-12-20,17
"Dầu DO 0,001S-V",2017-12-15,2024-12-26,40
"Dầu DO 0,005S-IV",2018-08-22,2019-01-01,10
Xăng RON 95-III,2018-08-22,2024-12-26,205
Xăng RON 95-IV,2018-08-22,2019-01-01,10


In [19]:
fuel_core = fuel_selected[
    (fuel_selected["mat_hang_clean"] == "Dầu DO 0,05S")
    |
    (
        (fuel_selected["mat_hang_clean"] == "Xăng RON 95")
        & (fuel_selected["ngay"] < "2019-01-01")
    )
    |
    (
        (fuel_selected["mat_hang_clean"] == "Xăng RON 95-III")
        & (fuel_selected["ngay"] >= "2019-01-01")
    )
].copy()
fuel_core["mat_hang_clean"].value_counts()

mat_hang_clean
Dầu DO 0,05S       279
Xăng RON 95-III    196
Xăng RON 95        105
Name: count, dtype: int64

In [20]:
fuel_core["fuel_type"] = fuel_core["mat_hang_clean"].replace({
    "Dầu DO 0,05S": "Diesel",
    "Xăng RON 95": "RON95",
    "Xăng RON 95-III": "RON95"
})
fuel_core["fuel_type"].value_counts()

fuel_type
RON95     301
Diesel    279
Name: count, dtype: int64

## Chuyển giá nhiên liệu về dữ liệu theo ngày

Giá xăng dầu được ghi nhận tại các thời điểm điều chỉnh. Mỗi mức giá được xem là có hiệu lực từ ngày điều chỉnh cho đến trước lần điều chỉnh tiếp theo. Vì vậy, giá được mở rộng theo ngày bằng phương pháp forward-fill trước khi tính giá trung bình theo tháng.

In [21]:
fuel_daily = (
    fuel_core[["ngay", "fuel_type", "gia_moi"]]
    .sort_values(["fuel_type", "ngay"])
    .set_index("ngay")
    .groupby("fuel_type")["gia_moi"]
    .resample("D")
    .ffill()
    .reset_index()
)
fuel_daily.head(15)

,fuel_type,ngay,gia_moi
0,Diesel,2012-08-01,20800.0
1,Diesel,2012-08-02,20800.0
2,Diesel,2012-08-03,20800.0
3,Diesel,2012-08-04,20800.0
4,Diesel,2012-08-05,20800.0
5,Diesel,2012-08-06,20800.0
6,Diesel,2012-08-07,20800.0
7,Diesel,2012-08-08,20800.0
8,Diesel,2012-08-09,20800.0
9,Diesel,2012-08-10,20800.0


In [24]:
fuel_monthly = (
    fuel_daily
    .assign(MonthYear=fuel_daily["ngay"].dt.to_period("M"))
    .groupby(["MonthYear", "fuel_type"])["gia_moi"]
    .mean()
    .round(2)
    .reset_index()
)

fuel_monthly.head(10)

,MonthYear,fuel_type,gia_moi
0,2012-08,Diesel,21298.39
1,2012-08,RON95,23158.06
2,2012-09,Diesel,21850.00
3,2012-09,RON95,24150.00
4,2012-10,Diesel,21850.00
5,2012-10,RON95,24150.00
6,2012-11,Diesel,21850.00
7,2012-11,RON95,23816.67
8,2012-12,Diesel,21811.29
9,2012-12,RON95,23650.00


In [26]:
fuel_monthly_wide = fuel_monthly.pivot(
    index="MonthYear",
    columns="fuel_type",
    values="gia_moi"
).reset_index()
fuel_monthly_wide.head(10)

fuel_type,MonthYear,Diesel,RON95
0,2012-08,21298.39,23158.06
1,2012-09,21850.00,24150.00
2,2012-10,21850.00,24150.00
3,2012-11,21850.00,23816.67
4,2012-12,21811.29,23650.00
5,2013-01,21550.00,23650.00
6,2013-02,21550.00,23650.00
7,2013-03,21550.00,23650.00
8,2013-04,21645.33,24340.33
9,2013-05,21770.00,24620.00
